In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from locallib.picarrodb import EU1_Conn, EU2_Conn
from data_extractor import ExtractionSpec, extract_multiple_data
from dotenv import load_dotenv
from apps_dal_dynamo.dynamodal import DynamoDalBase

import datetime
import time
import os

/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
import boto3
import dotenv
import io
import pandas as pd 
import os
import time
# # These should be set in the .env file:
#
# # Athena US variables
# AWS_ATHENA_US_PROFILE=Data-Analytics-Athena-User-xxxxxxxxxxxx
#
# # Athena Dynamo DB exports (analyzer data)
# AWS_ATHENA_DYNAMOEXPORTS_WG=picarro-inventory-wg
# AWS_ATHENA_DYNAMOEXPORTS_DB=dynamodb_exports
# AWS_ATHENA_DYNAMOEXPORTS_REGION=us-west-2
#
# # Athena handheld data
# AWS_ATHENA_HANDHELD_WG=handheld-prod
# AWS_ATHENA_HANDHELD_DB=handheld_measurements_data
# AWS_ATHENA_HANDHELD_REGION=us-east-1
dotenv.load_dotenv()
# Get AWS setup
AWS_ATHENA_US_PROFILE = os.getenv('AWS_ATHENA_US_PROFILE')
AWS_ATHENA_DYNAMOEXPORTS_WG = os.getenv('AWS_ATHENA_DYNAMOEXPORTS_WG')
AWS_ATHENA_DYNAMOEXPORTS_DB=os.getenv('AWS_ATHENA_DYNAMOEXPORTS_DB')
AWS_ATHENA_DYNAMOEXPORTS_REGION=os.getenv('AWS_ATHENA_DYNAMOEXPORTS_REGION')
AWS_ATHENA_HANDHELD_WG = os.getenv('AWS_ATHENA_HANDHELD_WG')
AWS_ATHENA_HANDHELD_DB = os.getenv('AWS_ATHENA_HANDHELD_DB')
AWS_ATHENA_HANDHELD_REGION = os.getenv('AWS_ATHENA_HANDHELD_REGION')
for var in [
    AWS_ATHENA_US_PROFILE,
    AWS_ATHENA_DYNAMOEXPORTS_WG,
    AWS_ATHENA_DYNAMOEXPORTS_DB,
    AWS_ATHENA_DYNAMOEXPORTS_REGION,
    AWS_ATHENA_HANDHELD_WG,
    AWS_ATHENA_HANDHELD_DB,
    AWS_ATHENA_HANDHELD_REGION,
]:
    assert var is not None, f"Variable {var} is not set"

In [4]:
# One-time setup: run from terminal if you get ProfileNotFound:
#   bash setup_athena_aws.sh
#
# Or verify your profile is configured:
import boto3

available = boto3.Session().available_profiles
print("AWS profiles on this machine:", available or "(none)")
print("Expected profile:", AWS_ATHENA_US_PROFILE)

if AWS_ATHENA_US_PROFILE not in available:
    raise RuntimeError(
        f"AWS profile '{AWS_ATHENA_US_PROFILE}' not found.\n"
        "Run: bash setup_athena_aws.sh\n"
        "You need your org's AWS SSO start URL (AWS access portal URL)."
    )
print("Profile found — OK")

AWS profiles on this machine: ['default', 'Data-Analytics-Athena-User-580855453261']
Expected profile: Data-Analytics-Athena-User-580855453261
Profile found — OK


In [5]:
session_dynamoexports = boto3.Session(
  profile_name=AWS_ATHENA_US_PROFILE, 
  region_name=AWS_ATHENA_DYNAMOEXPORTS_REGION
)
athena_dynamoexports = session_dynamoexports.client('athena')
s3_dynamoexports = session_dynamoexports.client('s3')

In [6]:
from athena_query import run_athena_query

In [9]:
print("Run Test Quer for Dynamo Exports")
# Create simple data query
query_dynamoexports = """
SELECT *
FROM surveyorproduction_measurement_parquet
WHERE analyzer_id = 'efd64983-a369-367c-8590-39fb2593d35a'
  AND year = 2021
  AND month = 9

  AND day = 15
LIMIT 10;
"""
out_loc_dynamoexports = run_athena_query(
    query=query_dynamoexports, 
    client=athena_dynamoexports,
    db=AWS_ATHENA_DYNAMOEXPORTS_DB, 
    wg=AWS_ATHENA_DYNAMOEXPORTS_WG
)

Run Test Quer for Dynamo Exports


In [10]:
bucket_dynamoexports, _, key_dynamoexports = out_loc_dynamoexports[len("s3://"):].partition("/")
obj_dynamoexports = s3_dynamoexports.get_object(Bucket=bucket_dynamoexports, Key=key_dynamoexports)
df_dynamoexports = pd.read_csv(io.BytesIO(obj_dynamoexports['Body'].read()))
print(f"Bucket: {bucket_dynamoexports}")
print(f"Key: {key_dynamoexports}\n")
print(df_dynamoexports.head())

Bucket: athena-query-results-abhishek
Key: c75d309d-538f-4635-b628-7c948548f486.csv

     epoch_time       ch4  c2h4   c2h6  co2  delta_ch4  h2o_percent  \
0  1.631668e+09  2.122033   0.0  0.009  NaN        NaN          NaN   
1  1.631668e+09  2.113178   0.0  0.009  NaN        NaN          NaN   
2  1.631668e+09  2.111477   0.0  0.009  NaN        NaN          NaN   
3  1.631668e+09  2.052954   0.0  0.009  NaN        NaN          NaN   
4  1.631668e+09  2.043238   0.0  0.009  NaN        NaN          NaN   

   wind_speed_lateral  hot_box_temperature  analyzer_mode  ...  \
0            0.625299            30.194391              5  ...   
1            0.503347            30.283163              5  ...   
2            0.805870            30.298273              5  ...   
3            0.159480            30.499414              5  ...   
4            0.244435            30.545704              5  ...   

   wind_speed_north  wind_direction_stddev  additional_fields  \
0         -0.275246       